# SmartWafer — Classification Training

Notebook untuk melatih dan membandingkan lima model supervised learning untuk klasifikasi `defect_label`:
- `0` = Tidak Defect
- `1` = Defect

Notebook ini menggunakan `preprocessing.py` yang sudah ada. Jalankan setelah EDA dan preprocessing awal selesai.

**Catatan penting dari hasil EDA:** dataset berisi 5.000 baris, tetapi distribusi target yang tercatat adalah 4.993 kelas 0 dan hanya 7 kelas 1. Ini merupakan ketidakseimbangan kelas yang sangat ekstrem. Metrik test dapat berubah drastis karena jumlah contoh defect sangat sedikit; hasil model harus dianggap eksperimen awal, bukan bukti performa produksi.

## 1. Import library dan konfigurasi

Jika ada library yang belum terpasang, instal di environment Python yang digunakan oleh Jupyter.

In [ ]:
from pathlib import Path
import sys
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    average_precision_score, make_scorer
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("scikit-learn:", sklearn.__version__)

## 2. Temukan `preprocessing.py` dan dataset

Sesuaikan `PROJECT_ROOT` atau `DATA_PATH` bila struktur folder proyek berbeda. Notebook ini mencoba mencari file pada folder notebook dan satu tingkat di atasnya.

In [ ]:
CURRENT_DIR = Path.cwd().resolve()

possible_roots = [
    CURRENT_DIR,
    CURRENT_DIR.parent,
]

PROJECT_ROOT = next(
    (p for p in possible_roots if (p / "preprocessing.py").exists()),
    CURRENT_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import preprocessing

print("Project root:", PROJECT_ROOT)
print("preprocessing.py:", PROJECT_ROOT / "preprocessing.py")

# Prioritaskan dataset featured hasil notebook preprocessing.
candidate_paths = [
    PROJECT_ROOT / "semiconductor_wafer_defect_featured.csv",
    PROJECT_ROOT / "data" / "processed" / "semiconductor_wafer_defect_featured.csv",
    PROJECT_ROOT / "data" / "processed" /"semiconductor_wafer_defect_clean.csv",
    PROJECT_ROOT / "data" / "raw" / "semiconductor_wafer_defect_dataset.csv",
]

DATA_PATH = next((p for p in candidate_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Dataset CSV tidak ditemukan. Letakkan semiconductor_wafer_defect_featured.csv "
        "di folder proyek, atau ubah DATA_PATH secara manual."
    )

print("Dataset:", DATA_PATH)

## 3. Muat data dan gunakan fungsi dari `preprocessing.py`

`feature_engineering()` membuat `electrical_power_mw` bila kolom tegangan dan arus tersedia, serta menghapus `wafer_id`. Fungsi tersebut belum menangani missing value; penanganan nilai kosong dilakukan di dalam pipeline model agar parameter imputasi dipelajari dari data training saja.

In [ ]:
df = preprocessing.load_data(DATA_PATH)
df = preprocessing.feature_engineering(df)

if "defect_label" not in df.columns:
    raise ValueError("Kolom target 'defect_label' tidak ditemukan.")

df = df.dropna(subset=["defect_label"]).copy()
df["defect_label"] = df["defect_label"].astype(int)

X, y = preprocessing.split_features_target(df)

print("Ukuran X:", X.shape)
print("Ukuran y:", y.shape)
print("\nKolom fitur:", X.columns.tolist())
print("\nDistribusi target:")
display(y.value_counts(dropna=False).sort_index().rename("jumlah").to_frame())
print("\nProporsi target:")
display(y.value_counts(normalize=True, dropna=False).sort_index().rename("proporsi").to_frame())

if set(y.unique()) != {0, 1}:
    print("PERINGATAN: target tidak berisi tepat dua kelas {0, 1}. Periksa label dataset.")

## 4. Periksa ketidakseimbangan kelas

Jangan hanya mengandalkan accuracy. Jika model selalu memprediksi kelas mayoritas, accuracy dapat terlihat sangat tinggi meskipun model gagal menemukan defect.

In [ ]:
class_counts = y.value_counts()
minority_count = int(class_counts.min())

ax = class_counts.sort_index().plot(kind="bar", rot=0, figsize=(6, 4))
ax.set_title("Distribusi defect_label")
ax.set_xlabel("defect_label (0 = Tidak Defect, 1 = Defect)")
ax.set_ylabel("Jumlah sampel")
plt.tight_layout()
plt.show()

print(f"Jumlah kelas minoritas: {minority_count}")
if minority_count < 30:
    print(
        "PERINGATAN: sampel kelas minoritas sangat sedikit. "
        "Satu test split atau cross-validation tidak cukup untuk menyimpulkan performa model dengan kuat."
    )

## 5. Bagi data menjadi training dan testing

Data test ditahan untuk evaluasi akhir. `stratify=y` menjaga proporsi kelas sebisa mungkin. Karena jumlah kelas defect sangat kecil, test set mungkin hanya berisi satu atau dua sampel defect; interpretasikan metrik dengan hati-hati.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Training:", X_train.shape, "| distribusi:", y_train.value_counts().to_dict())
print("Testing :", X_test.shape, "| distribusi:", y_test.value_counts().to_dict())

if y_train.value_counts().min() < 2 or y_test.value_counts().min() < 1:
    raise ValueError(
        "Jumlah kelas terlalu sedikit untuk split stratified ini. "
        "Tambahkan data defect atau tinjau strategi evaluasi."
    )

## 6. Siapkan preprocessing pipeline

Fitur numerik akan diimputasi dengan median dan distandardisasi. Fitur kategorikal seperti `process_step` akan diimputasi dengan modus lalu di-one-hot encode. Seluruh langkah dipasang di pipeline untuk menghindari data leakage.

In [ ]:
numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ],
    remainder="drop"
)

print("Fitur numerik:", numeric_features)
print("Fitur kategorikal:", categorical_features)

## 7. Definisikan lima model supervised learning

Model yang dibandingkan:
1. Logistic Regression
2. Random Forest
3. Extra Trees
4. Support Vector Machine (SVM)
5. Decision Tree

`class_weight="balanced"` digunakan pada model yang mendukungnya untuk memberi bobot lebih besar pada kelas minoritas. Ini bukan pengganti penambahan data defect dan tidak menjamin hasil yang baik.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "Extra Trees": ExtraTreesClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "SVM": SVC(
        kernel="rbf",
        class_weight="balanced",
        probability=True,
        random_state=RANDOM_STATE
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=RANDOM_STATE,
        min_samples_leaf=2
    )
}

pipelines = {
    name: Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", estimator)
    ])
    for name, estimator in models.items()
}

print("Jumlah model:", len(pipelines))

## 8. Latih dan bandingkan model

Metrik utama yang ditampilkan adalah precision, recall, dan F1 untuk kelas defect (`pos_label=1`), ditambah F1 macro dan average precision jika model mendukung `predict_proba` atau `decision_function`.

Karena jumlah defect hanya tujuh pada hasil EDA saat ini, metrik per kelas akan sangat tidak stabil. Periksa jumlah true positive, false negative, dan false positive pada confusion matrix.

In [ ]:
results = []
fitted_models = {}
predictions = {}

for name, pipeline in pipelines.items():
    print(f"Training: {name}")
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    fitted_models[name] = pipeline
    predictions[name] = y_pred

    row = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_defect": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
        "Recall_defect": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1_defect": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
        "F1_macro": f1_score(y_test, y_pred, average="macro", zero_division=0),
    }

    try:
        if hasattr(pipeline, "predict_proba"):
            scores = pipeline.predict_proba(X_test)[:, 1]
            row["Average_precision"] = average_precision_score(y_test, scores)
        elif hasattr(pipeline, "decision_function"):
            scores = pipeline.decision_function(X_test)
            row["Average_precision"] = average_precision_score(y_test, scores)
        else:
            row["Average_precision"] = np.nan
    except Exception:
        row["Average_precision"] = np.nan

    results.append(row)

results_df = pd.DataFrame(results).sort_values(
    ["F1_defect", "Recall_defect", "F1_macro"],
    ascending=False
).reset_index(drop=True)

display(results_df)

## 9. Confusion matrix dan classification report

Periksa setiap model, bukan hanya model yang berada di urutan pertama. Dengan kelas defect yang sangat sedikit, satu kesalahan prediksi dapat mengubah recall secara besar.

In [ ]:
for name, y_pred in predictions.items():
    print("=" * 70)
    print(name)
    print(classification_report(
        y_test,
        y_pred,
        labels=[0, 1],
        target_names=["Tidak Defect (0)", "Defect (1)"],
        zero_division=0
    ))

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["Tidak Defect (0)", "Defect (1)"]
    )
    disp.plot(values_format="d")
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()

## 10. Pilih model kandidat dan simpan artefak

Notebook memilih kandidat berdasarkan F1 kelas defect, lalu recall defect dan F1 macro sebagai tie-breaker. Dengan dataset yang sangat tidak seimbang, jangan menerima pilihan otomatis ini tanpa meninjau confusion matrix dan kebutuhan bisnis.

Model disimpan bersama preprocessing pipeline dalam satu objek `Pipeline`, sehingga transformasi fitur yang sama dapat digunakan pada prediksi berikutnya.

In [ ]:
best_name = results_df.iloc[0]["Model"]
best_model = fitted_models[best_name]

print("Kandidat model berdasarkan metrik test:", best_name)
display(results_df.iloc[[0]])

In [ ]:
MODEL_DIR = PROJECT_ROOT / "models" / "classification"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

model_path = MODEL_DIR / "best_model.joblib"
metrics_path = MODEL_DIR / "model_comparison.csv"
metadata_path = MODEL_DIR / "model_metadata.json"

joblib.dump(best_model, model_path)
results_df.to_csv(metrics_path, index=False)

metadata = {
    "project": "SmartWafer",
    "target_column": "defect_label",
    "class_mapping": {"0": "Tidak Defect", "1": "Defect"},
    "selected_model": best_name,
    "feature_columns": X.columns.tolist(),
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "sklearn_version": sklearn.__version__,
    "warning": (
        "EDA menunjukkan hanya 7 sampel defect dari 5000 baris. "
        "Evaluasi sangat tidak pasti; validasi dengan lebih banyak data defect sebelum produksi."
    )
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Model tersimpan:", model_path)
print("Metrik tersimpan:", metrics_path)
print("Metadata tersimpan:", metadata_path)

## 11. Uji pemuatan model tersimpan

Jalankan cell ini untuk memastikan model dapat dimuat kembali dan menerima satu baris fitur dengan kolom yang sama.

In [ ]:
loaded_model = joblib.load(model_path)

sample = X_test.iloc[[0]].copy()
sample_prediction = loaded_model.predict(sample)[0]

print("Prediksi:", int(sample_prediction))
print("Label aktual:", int(y_test.iloc[0]))

if hasattr(loaded_model, "predict_proba"):
    print("Probabilitas kelas:", loaded_model.predict_proba(sample))

## 12. Catatan sebelum integrasi FastAPI

- Backend harus mengirim fitur dengan nama kolom dan tipe data yang sesuai dengan training.
- `wafer_id` tidak digunakan sebagai fitur model.
- `electrical_power_mw` dibuat oleh `preprocessing.feature_engineering()`. Pastikan backend juga menerapkan fungsi yang sama sebelum prediksi.
- Model disimpan sebagai `best_model.joblib`; muat dengan `joblib.load()` di backend.
- Gunakan environment/library version yang kompatibel dengan saat model disimpan.
- Jangan klaim model siap produksi hanya berdasarkan hasil ini. Dengan hanya tujuh sampel defect pada dataset EDA, kumpulkan data defect tambahan dan evaluasi ulang. Pertimbangkan split berdasarkan wafer/batch jika beberapa baris berasal dari kelompok proses yang sama.